# Average Multiple Days Together

## Imports and Versions

In [1]:
from pygsdata import GSData
from pathlib import Path

from edges_cal import modelling as mdl

#from edges_analysis.averaging.combiners import average_over_times
from edges_analysis.datamodel import add_model
from edges_analysis.filters.filters import rfi_model_filter
from edges_cal.alanmode import read_spec_txt
from pygsdata import plots
import matplotlib.pyplot as plt
import numpy as np
from edges_pipelines import utils
from edges_analysis.filters import filters
from pygsdata import GSFlag

from pygsdata.register import gsregister
from edges_analysis.filters.filters import gsdata_filter
from typing import Literal
from astropy import units as u
from edges_analysis.averaging import averaging

from edges_analysis.averaging.combiners import lst_average
from edges_analysis.datamodel import add_model
from edges_cal.modelling.models import PhysicalLin

ModuleNotFoundError: No module named 'edges_pipelines'

In [ ]:
plt.style.use("default")

In [ ]:
utils.print_versions()


## Parameters and Data Loading

In [ ]:
datadir: str = "/data4/vydula/edges/edges3-data-analysis/scripts/prefect-10days/prefect-outputs"
nightly_files: list[str] = [Path(f"{datadir}/2023_{day}.finalspec.gsh5") for day in range(300, 309)]


In [ ]:
datadir = Path(datadir)

In [ ]:
data = [GSData.from_file(d) for d in nightly_files]

In [ ]:
data_with_model = [add_model(d, nsamples_strategy='flagged-nsamples-uniform',
                             model= PhysicalLin(
                                 spectral_index =-2.55, 
                                 f_center= 85, 
                                 n_terms =5)) for d in data]

In [ ]:
fig, ax = plt.subplots(len(data), 1, sharex=True, constrained_layout=True, figsize=(5, 20))
for i in range(len(data)):
    plots.plot_waterfall(data_with_model[i], attribute='data',vmin=0, vmax=10000, ax= ax[i], xlab= False) 

    

## Analysis and Averaging

### Model the spectra so we can average residuals only

When we perform the average over nights, we average the models and residuals separately (and the residuals receive frequency-dependent weights while the models do not). First, we model each night here.

### Some Filters

Our first filter is an RMS filter (just thresholding a whole night based on its RMS to the fitted model)

In [ ]:
@gsregister("filter")
@gsdata_filter()
def rms_filter(
    data: GSData,
    threshold: float,
    freq_range: float = (0.0, np.inf),
    nsamples_strategy: Literal[
        "flagged-nsamples",
        "flags-only",
        "flagged-nsamples-uniform",
        "nsamples-only",
    ] = "flagged-nsamples",
    model: mdl.Model | None = None,
) -> bool:
    """Filter integrations based on the rms of the residuals.
    
    Parameters
    ----------
    data : GSData
        The data to be filtered.
    threshold
        The threshold at which to flag integrations.
    freq_range : float, optional
        The frequency range to use in calculating the RMS.
    nsamples_strategy : str, optional
        The strategy to use when defining the weights of each sample. Defaults to
        'flagged-nsamples'. The choices are:
        - 'flagged-nsamples': Use the flagged nsamples (i.e. set nsamples at flagged
            data to zero, otherwise use nsamples)
        - 'flags-only': Use the flags only (i.e. set nsamples at flagged data to
            zero, otherwise use 1)
        - 'flagged-nsamples-uniform': Use the flagged nsamples (i.e. set nsamples at
            flagged data to zero, and keep zero-samples as zero, otherwise use 1)
        - 'nsamples-only': Use the nsamples only (don't set nsamples at flagged
            data to zero)
    model : Model, optional
        A model to be used to fit each integration. Not required if a model
        already exists on the data.
    """
    if freq_range[0]*u.MHz > data.freqs.min() or freq_range[1]*u.MHz < data.freqs.max():
        data = flag_frequency_ranges(data=data, freq_ranges=[freq_range], invert=True)

    if data.residuals is None:
        if model is None:
            raise ValueError(
                "Cannot perform object rms filter without residuals or a model."
            )
        data = add_model(data=data, model=model, nsamples_strategy=nsamples_strategy)
        
    shp = (-1, data.nfreqs)
    if nsamples_strategy == "flagged-nsamples":
        w = data.flagged_nsamples.reshape(shp)
    elif nsamples_strategy == "flags-only":
        w = (~data.complete_flags.reshape(shp)).astype(float)
    elif nsamples_strategy == "flagged-nsamples-uniform":
        w = (data.flagged_nsamples > 0).astype(float).reshape(shp)
    elif nsamples_strategy == "nsamples-only":
        w = np.nsamples.reshape(shp)
    
    rms = np.sqrt(
        averaging.weighted_mean(data=data.residuals.reshape(shp)**2, weights=w, axis=-1)[0]
    )

    return GSFlag(
        flags=(rms > threshold).reshape(data.data.shape[:-1]),
        axes=("load", "pol", "time",),
    )


In [ ]:
#rms_data = [rms_filter(d, threshold=0.17, nsamples_strategy='flagged-nsamples-uniform') for d in data_with_model]

In [ ]:
# #assert np.sum(~rms_data.flags['rms_filter'].flags[0,0] ^ alanfilt[:, 1].astype(bool))==0
# rms_data = filters.prune_flagged_integrations(rms_data)
# print(rms_data.ntimes)

Based on the model already fit, we perform simple RFI-flagging, where we flag any channel whose residual is larger than a threshold multiplied by the RMS of the residuals that night (over frequency).

In [ ]:
@filters.gsdata_filter()
def rms_rfi_filter(data: GSData, threshold):
    
    
    ns = (data.nsamples[0,0] > 0).astype(float)

    sumn = np.sum(ns, axis=1)
    sumres = np.sum(data.residuals[0,0]**2 * ns, axis=1)
    rms = np.sqrt(np.where(sumn > 0, sumres / sumn, 0))[None, None, :, None]

    return GSFlag(flags=data.residuals > rms * threshold, axes=("load", "pol", "time", "freq"))

In [ ]:
#[np.all(d.complete_flags) for d in rms_data]

In [ ]:

filt_data = [rms_rfi_filter(d, threshold=1.9) for d in data_with_model]
no_filt_data = data_with_model

In [ ]:
fig, ax = plt.subplots(len(filt_data), 1, sharex=True, constrained_layout=True, figsize=(5, 20))
for i in range(len(filt_data)):
    plots.plot_waterfall(filt_data[i], attribute='data',vmin=0, vmax=10000, ax= ax[i], xlab= False)

### Average all nights

In [ ]:
import warnings

In [ ]:
@gsregister("gather")
def lst_average_test(
    *objs,
    use_nsamples: bool = True,
    nsamples_strategy: Literal[
        "flagged-nsamples",
        "flags-only",
        "flagged-nsamples-uniform",
        "nsamples-only",
    ] = "flagged-nsamples",
    use_flags: bool = True,
    use_resids: bool | None = None,
) -> GSData:
    """Average multiple objects together using their flagged weights."""
    if any(not np.allclose(obj.lsts.hour, objs[0].lsts.hour) for obj in objs[1:]):
        raise ValueError("All objects must have the same LST array to average them.")

    if any(obj.data.shape != objs[0].data.shape for obj in objs[1:]):
        raise ValueError("All objects must have the same shape to average them.")

    if use_nsamples:
        nsamples = [obj.nsamples for obj in objs]
    elif use_flags:
        nsamples = [(~(obj.flagged_nsamples == 0)).astype(float) for obj in objs]
    else:
        nsamples = [1] * len(objs)

    #tot_nsamples = np.nansum(nsamples, axis=0)

    if use_resids is None:
        use_resids = all(obj.residuals is not None for obj in objs)

    if use_resids and any(obj.residuals is None for obj in objs):
        raise ValueError("One or more of the input objects has no residuals.")

    if nsamples_strategy == "flagged-nsamples":
        w = [obj.flagged_nsamples for obj in objs]
        _n = w
    elif nsamples_strategy == "flags-only":
        w = [(~obj.complete_flags).astype(float) for obj in objs]
        _n = [obj.flagged_nsamples for obj in objs]
    elif nsamples_strategy == "flagged-nsamples-uniform":
        w = [(obj.flagged_nsamples > 0).astype(float) for obj in objs]
        _n = [obj.flagged_nsamples for obj in objs]
    elif nsamples_strategy == "nsamples-only":
        w = [obj.nsamples for obj in objs]
        _n = w
    else:
        raise ValueError(
            f"Invalid nsamples_strategy: {nsamples_strategy}. Must be one of "
            "'flagged-nsamples', 'flags-only', 'flagged-nsamples-uniform' or "
            "'nsamples-only'"
        )

    ntot = np.sum(w, axis=-2)
    tot_nsamples_all = np.array(_n)

    print('tot_nsamples',tot_nsamples_all.shape)


    
    if use_resids:
        residuals = np.nansum(
            [obj.residuals * n for obj, n in zip(objs, nsamples)], axis=0
        )

        

        for i in range(len(objs)):
            tot_nsamples = tot_nsamples_all[i]

            residuals[tot_nsamples > 0] /= tot_nsamples[tot_nsamples > 0]
        tot_model = np.nansum([obj.model for obj in objs], axis=0)
        tot_obj = len(objs) - sum([np.all(np.isnan(obj.model), axis=3) for obj in objs])
        print('tot_obj',tot_obj.shape)
        print('ntot',ntot.shape)
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore")
            tot_model /= (tot_obj)[..., None]
        #logger.debug(f"After combining sum(residuals): {np.nansum(residuals)}")
        final_data = tot_model + residuals
    else:
        final_data = np.nansum([obj.data * n for obj, n in zip(objs, nsamples)], axis=0)
        final_data[tot_nsamples > 0] /= tot_nsamples[tot_nsamples > 0]
        residuals = None

    return objs[0].update(
        data=final_data,
        residuals=residuals,
        nsamples=tot_nsamples,
        flags={},
    )


In [ ]:
@gsregister("gather")
def lst_average_test(
    *objs,
    use_nsamples: bool = True,
    nsamples_strategy: Literal[
        "flagged-nsamples",
        "flags-only",
        "flagged-nsamples-uniform",
        "nsamples-only",
    ] = "flagged-nsamples",
    use_flags: bool = True,
    use_resids: bool | None = None,
    fill_value: float = 0.0,

) -> GSData:
    """Average multiple objects together using their flagged weights."""
    if any(not np.allclose(obj.lsts.hour, objs[0].lsts.hour) for obj in objs[1:]):
        raise ValueError("All objects must have the same LST array to average them.")

    if any(obj.data.shape != objs[0].data.shape for obj in objs[1:]):
        raise ValueError("All objects must have the same shape to average them.")

    if use_nsamples:
        nsamples = [obj.nsamples for obj in objs]
    elif use_flags:
        nsamples = [(~(obj.flagged_nsamples == 0)).astype(float) for obj in objs]
    else:
        nsamples = [1] * len(objs)

    #tot_nsamples = np.nansum(nsamples, axis=0)

    if use_resids is None:
        use_resids = all(obj.residuals is not None for obj in objs)

    if use_resids and any(obj.residuals is None for obj in objs):
        raise ValueError("One or more of the input objects has no residuals.")

    if nsamples_strategy == "flagged-nsamples":
        w = [obj.flagged_nsamples for obj in objs]
        _n = w
    elif nsamples_strategy == "flags-only":
        w = [(~obj.complete_flags).astype(float) for obj in objs]
        _n = [obj.flagged_nsamples for obj in objs]
    elif nsamples_strategy == "flagged-nsamples-uniform":
        w = [(obj.flagged_nsamples > 0).astype(float) for obj in objs]
        _n = [obj.flagged_nsamples for obj in objs]
    elif nsamples_strategy == "nsamples-only":
        w = [obj.nsamples for obj in objs]
        _n = w
    else:
        raise ValueError(
            f"Invalid nsamples_strategy: {nsamples_strategy}. Must be one of "
            "'flagged-nsamples', 'flags-only', 'flagged-nsamples-uniform' or "
            "'nsamples-only'"
        )

    ntot = np.sum(w, axis=-2)
    tot_nsamples = np.sum(_n, axis=-2)

    print('tot_nsamples',tot_nsamples.shape)
    print('ntot',ntot.shape)
    print('obj', objs[0].data.shape)


    if use_resids:
        sum_resids = np.sum([obj.residuals * _w for obj, _w in zip(objs, w)], axis=-2)
        print('sum_resids', sum_resids.shape)

        mean_resids = sum_resids / ntot
        mean_model = np.mean([obj.model for obj in objs], axis=-2)
        new_data = mean_model + mean_resids
        
        residuals = np.nansum(
            [obj.residuals * n for obj, n in zip(objs, nsamples)], axis=0
        )
        
    else:
        sum_data = np.sum([obj.data * _w for obj, _w in zip(objs, w)], axis=-2)
        new_data = sum_data / ntot
        residuals = None

    new_data[np.isnan(new_data)] = fill_value
    
    print('new_data', new_data.shape)
    
#    if use_resids:
#         residuals = np.nansum(
#             [obj.residuals * n for obj, n in zip(objs, nsamples)], axis=0
#         )

        

#         for i in range(len(objs)):
#             tot_nsamples = tot_nsamples_all[i]

#             residuals[tot_nsamples > 0] /= tot_nsamples[tot_nsamples > 0]
#         tot_model = np.nansum([obj.model for obj in objs], axis=0)
#         tot_obj = len(objs) - sum([np.all(np.isnan(obj.model), axis=3) for obj in objs])
#         print('tot_obj',tot_obj.shape)
#         print('ntot',ntot.shape)
#         with warnings.catch_warnings():
#             warnings.filterwarnings("ignore")
#             tot_model /= (tot_obj)[..., None]
#         #logger.debug(f"After combining sum(residuals): {np.nansum(residuals)}")
#         final_data = tot_model + residuals
#     else:
#         final_data = np.nansum([obj.data * n for obj, n in zip(objs, nsamples)], axis=0)
#         final_data[tot_nsamples > 0] /= tot_nsamples[tot_nsamples > 0]
#         residuals = None

    return objs[0].update(
        data=new_data,
        residuals=residuals,
        nsamples=tot_nsamples,
        flags={},
    )


In [ ]:
filt_data[0].freqs.shape

In [ ]:
avg_data = lst_average_test(*filt_data, use_nsamples=False, use_flags=True, use_resids=True)
no_avg_data = lst_average_test(*no_filt_data, use_nsamples=False, use_flags=True, use_resids=True)


In [ ]:
avg_data.nsamples

In [ ]:
plt.imshow((filt_data[0].complete_flags).astype(float)[0,0] - (no_filt_data[0].complete_flags).astype(float)[0,0], aspect='auto')

In [ ]:
plots.plot_waterfall(avg_data,attribute='data',vmin=0, vmax=10000)

### Last RFI Filter

In [ ]:
final_data = add_model(avg_data, model=mdl.LinLog(n_terms=5))
#no_final_data = add_model(no_avg_data, model=mdl.LinLog(n_terms=5))

## Inspect Final Results

In [ ]:
plots.plot_waterfall(final_data, attribute='data',vmin=0, vmax=10000, title='10 day average data')

In [ ]:
plots.plot_waterfall(final_data, attribute='residuals', vmin=0, vmax=4,title='10 day average residuals')

In [ ]:
(final_data.residuals[0,0]-no_final_data.residuals[0,0])

In [ ]:
np.where(final_data.residuals == np.max(final_data.residuals))

In [ ]:
plt.plot(avg_data.data[0,0,29]-avg_data.model[0,0,29])

In [ ]:
plt.plot(final_data.model[0,0,29])

In [ ]:
fig, ax = plt.subplots(2, 1, sharex=True, figsize=(10, 8), constrained_layout=True)

#ax[0].plot(final_data.freqs, np.where(final_data.flagged_nsamples[0,0,0]>0, final_data.data[0,0,0], np.nan), label='edges-analysis')
ax[0].plot(avg_data.freqs,avg_data.flagged_nsamples[0,0,0], label='edges-analysis')

ax[0].legend()

ax[0].text(0.95, 0.9, "Spectrum", transform=ax[0].transAxes, ha='right', fontweight='bold')

for i in range(2):
    ax[i].set_ylabel("Temperature [K]")
    
ttmin = avg_data.times.min().datetime.timetuple()
ttmax = avg_data.times.max().datetime.timetuple()

fig.suptitle(f"Final Averaged Spectrum: {avg_data.ntimes} days [{ttmin.tm_year}:{ttmin.tm_yday:>03} -- {ttmax.tm_year}:{ttmax.tm_yday:>03}]")

ax[1].plot(final_data.freqs, np.where(final_data.flagged_nsamples[0,0,0]>0, avg_data.residual, np.nan))

ax[1].text(0.95, 0.9, "FG Residuals (LinLog, 5-term)", transform=ax[1].transAxes, ha='right', fontweight='bold')




We ensure that we have all the same flags as the C-code:

In [ ]:
#assert np.sum((final_data.flagged_nsamples > 0) ^ alan['weight'].astype(bool))==0

## Write out the data

In [ ]:
final_data.write_gsh5(Path(datadir) / "averaged_spectrum.gsh5");